# NullVector LangGraph QA Agent — PostgreSQL Backend

This notebook builds a generative QA agent over a NullVector corpus stored in PostgreSQL.

**Prerequisite:** Run `03_nullvector_postgres_unified.ipynb` first to populate the Postgres database with acquisition, tree, and retrieval artifacts. This notebook resolves the concrete manifest refs for the PDF textbook runs from PostgreSQL before loading the corpus.

**Requirements:**
- A reachable PostgreSQL database with NullVector artifacts from notebook 03
- Inline Groq credentials are configured below through `GROQ_API_KEYS`; the agent rotates keys on every LLM call
- Optional model override via `NULLVECTOR_LLM_MODEL` (defaults to `meta-llama/llama-4-scout-17b-16e-instruct`)
- Optional: `NULLVECTOR_POSTGRES_CONNINFO` (defaults to `postgresql://REDACTED_DB_CRED@localhost:5432/nullvector`)

PostgreSQL artifact refs require a concrete `document_id`; wildcard refs such as `pg://retrieval/<run_id>/*/run-index.json` are not valid.


## Section 1 — Dependencies

Install the external LangGraph and LangChain packages. These are not NullVector dependencies — they are optional agent-framework integrations.

In [14]:
!uv pip install -q "langgraph>=0.2" "langchain-openai>=0.1" "langchain-core>=0.2"

## Section 2 — Load Corpus from PostgreSQL

We reuse the PDF run IDs from notebook 03 to load the textbook corpus and node cards. The notebook first resolves the retrieval and tree run rows from PostgreSQL, then follows their concrete `manifest_ref` values to load the corpus artifacts.

The agent keeps its Groq rotation helpers inline in the notebook so every model call can advance the key ring without depending on extra repo modules.

NullVector runtime observability is also configured here up front so retrieval tool calls emit the same human-readable progress lines and JSONL events as notebook 03.

In [15]:
from __future__ import annotations

import json
import os
import re
import textwrap
from collections import Counter
from typing import Annotated, TypedDict

from nullvector.domain import NodeCard
from nullvector.observability import (
    DEFAULT_OBSERVABILITY_JSONL_PATH,
    configure_default_runtime_observability,
)
from nullvector.domain.retrieval import RetrievalUnitType
from nullvector.retrieval import (
    QueryPlanner,
    RetrievalRanker,
    RetrievalService,
    load_retrieval_corpus,
    load_retrieval_manifest,
)
from nullvector.storage import PostgresStorageConfig, build_document_store

try:
    import psycopg
    from psycopg.rows import dict_row
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "Notebook 04 requires the postgres extra so it can resolve concrete run metadata from PostgreSQL. "
        "Install it with `uv sync --extra postgres` or `uv pip install 'psycopg[binary]>=3.1,<4'`."
    ) from exc

POSTGRES_CONNINFO = os.environ.get(
    "NULLVECTOR_POSTGRES_CONNINFO",
    "postgresql://REDACTED_DB_CRED@localhost:5432/app",
)
POSTGRES_SCHEMA = os.environ.get("NULLVECTOR_POSTGRES_SCHEMA", "public")
GROQ_DEFAULT_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"
AGENT_MODEL = os.environ.get("NULLVECTOR_LLM_MODEL", GROQ_DEFAULT_MODEL)
GROQ_BASE_URL = "https://api.groq.com/openai/v1"


def parse_groq_api_keys(raw_keys: str) -> tuple[str, ...]:
    keys = tuple(part.strip() for part in raw_keys.split(",") if part.strip())
    if not keys:
        raise ValueError("GROQ_API_KEYS must contain at least one non-empty key.")
    return keys


def groq_key_label(slot: int, key: str) -> str:
    suffix = key[-4:] if len(key) >= 4 else key
    return f"slot-{slot:02d} (gsk_...{suffix})"


GROQ_API_KEYS_INLINE = ",".join(
    (
        "REDACTED_GROQ_KEY",
        "REDACTED_GROQ_KEY",
        "REDACTED_GROQ_KEY",
        "REDACTED_GROQ_KEY",
        "REDACTED_GROQ_KEY",
    )
)
GROQ_API_KEYS = parse_groq_api_keys(GROQ_API_KEYS_INLINE)
GROQ_MASKED_KEYS = tuple(
    groq_key_label(slot, key) for slot, key in enumerate(GROQ_API_KEYS, start=1)
)
groq_key_cursor = 0

pg_config = PostgresStorageConfig(conninfo=POSTGRES_CONNINFO, schema=POSTGRES_SCHEMA)
store = build_document_store(pg_config)
OBSERVABILITY_JSONL_PATH = os.environ.get(
    "NULLVECTOR_OBSERVABILITY_JSONL_PATH",
    DEFAULT_OBSERVABILITY_JSONL_PATH,
)
runtime_logger = configure_default_runtime_observability()

# Run IDs matching notebook 03's PDF textbook flow
PDF_RETRIEVAL_RUN_ID = "cookbook-pdf-retrieval"
PDF_TREE_RUN_ID = "cookbook-pdf-tree"
TARGET_SOURCE_LABEL = "903000608.pdf"
RUN_TABLE_BY_TYPE = {
    "retrieval": "retrieval_runs",
    "tree": "tree_runs",
}


def next_groq_key_selection() -> tuple[str, str]:
    global groq_key_cursor
    index = groq_key_cursor % len(GROQ_API_KEYS)
    groq_key_cursor += 1
    return GROQ_API_KEYS[index], GROQ_MASKED_KEYS[index]


def validate_postgres_identifier(value: str, *, label: str) -> str:
    if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value) is None:
        raise ValueError(
            f"{label} must match ^[A-Za-z_][A-Za-z0-9_]*$ for notebook-local SQL resolution."
        )
    return value


def resolve_postgres_run_record(run_type: str, run_id: str) -> dict[str, object]:
    table_name = RUN_TABLE_BY_TYPE.get(run_type)
    if table_name is None:
        raise ValueError(f"Unsupported run_type for notebook lookup: {run_type!r}")
    schema_name = validate_postgres_identifier(
        POSTGRES_SCHEMA,
        label="NULLVECTOR_POSTGRES_SCHEMA",
    )
    with psycopg.connect(POSTGRES_CONNINFO, row_factory=dict_row) as conn:
        row = conn.execute(
            f"""
            SELECT run_id, document_id, status, artifact_root, manifest_ref
            FROM {schema_name}.{table_name}
            WHERE run_id = %s
            """,
            (run_id,),
        ).fetchone()
    if row is None:
        raise RuntimeError(
            f"{run_type} run {run_id!r} was not found in {schema_name}.{table_name}. "
            "Run 03_nullvector_postgres_unified.ipynb first or update the run ids in this notebook."
        )
    record = dict(row)
    status = str(record.get("status") or "")
    if status != "succeeded":
        raise RuntimeError(
            f"{run_type} run {run_id!r} is not ready yet (status={status!r})."
        )
    manifest_ref = record.get("manifest_ref")
    if not manifest_ref:
        raise RuntimeError(
            f"{run_type} run {run_id!r} is missing manifest_ref in {schema_name}.{table_name}."
        )
    return record


def load_tree_node_cards(tree_manifest: dict[str, object]) -> tuple[NodeCard, ...]:
    node_cards_path = tree_manifest.get("node_cards_path")
    if not node_cards_path:
        raise RuntimeError("tree manifest is missing node_cards_path")
    node_cards_payload = store.read_json_artifact(node_cards_path)
    return tuple(NodeCard.model_validate_json(json.dumps(item)) for item in node_cards_payload)


retrieval_run = resolve_postgres_run_record("retrieval", PDF_RETRIEVAL_RUN_ID)
DOCUMENT_ID = str(retrieval_run["document_id"])
RETRIEVAL_MANIFEST_REF = str(retrieval_run["manifest_ref"])
retrieval_manifest = load_retrieval_manifest(RETRIEVAL_MANIFEST_REF, storage=pg_config)
corpus = load_retrieval_corpus(retrieval_manifest.corpus_path, storage=pg_config)

tree_run = resolve_postgres_run_record("tree", PDF_TREE_RUN_ID)
TREE_DOCUMENT_ID = str(tree_run["document_id"])
if TREE_DOCUMENT_ID != DOCUMENT_ID:
    raise RuntimeError(
        "Notebook 04 expected the tree and retrieval runs to point at the same document, but got "
        f"retrieval={DOCUMENT_ID!r} and tree={TREE_DOCUMENT_ID!r}."
    )
TREE_MANIFEST_REF = str(tree_run["manifest_ref"])
tree_manifest = store.read_json_artifact(TREE_MANIFEST_REF)
node_cards = load_tree_node_cards(tree_manifest)
titles_by_id = {card.node_id: card.title for card in node_cards}

counts = Counter(unit.unit_type.value for unit in corpus.units)
print(f"Loaded corpus: {len(corpus.units)} units from document {DOCUMENT_ID}")
print(f"Source label: {TARGET_SOURCE_LABEL}")
print(f"Unit types: {dict(sorted(counts.items()))}")
print(f"Retrieval manifest ref: {RETRIEVAL_MANIFEST_REF}")
print(f"Tree manifest ref: {TREE_MANIFEST_REF}")
print(f"Node titles: {[card.title for card in node_cards]}")
print(f"Groq agent model: {AGENT_MODEL}")
print(f"Groq keys: {list(GROQ_MASKED_KEYS)}")
print(f"Observability JSONL: {OBSERVABILITY_JSONL_PATH}")


Loaded corpus: 3388 units from document 798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896
Source label: 903000608.pdf
Unit types: {'node_text': 3000, 'page_text': 140, 'table': 46, 'unassigned_span': 1, 'unresolved_visual': 4, 'visual': 197}
Retrieval manifest ref: pg://retrieval/cookbook-pdf-retrieval/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/manifest.json
Tree manifest ref: pg://tree/cookbook-pdf-tree/798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896/manifest.json
Node titles: ['The Constitution of India', 'Chapter IV A', 'Fundamental Duties', 'Sachin Mehta', 'HARSHWARDHAN OFFSET,KOLHAPUR', 'N/PB/2022-23/1,10,000', 'Sets', '• Equal sets, subset \t •  Universal set', '• Intersection and Union of sets', '• Number of elements in a set', 'Flower', 'Sets', 'Q ={ q', 'p  such that  p and q are integers', 'where q is a non-zero number.’', 'B = { -3, 3}', 'C = { 5, 10, 15, 20, 25}', 'Rule Method', 'M = {1, 8, 27, 64, 125.......}', 'Practic

## Section 3 — NullVector Retrieval Tools

Two tools expose NullVector retrieval to the LangGraph agent:
- `search_document` — runs `RetrievalService.search()` and formats results
- `get_page_text` — returns the full text of a specific logical page

In [16]:
from langchain_core.tools import tool

planner = QueryPlanner()
ranker = RetrievalRanker()
retrieval_service = RetrievalService(planner, ranker, logger=runtime_logger, storage=pg_config)


@tool
def search_document(query: str, limit: int = 5) -> str:
    """Search the NullVector document corpus for relevant content.

    Use this tool to find evidence before answering questions.
    Returns ranked retrieval hits with page numbers and excerpts.
    """
    hits = retrieval_service.search(corpus=corpus, query=query, limit=limit)
    if not hits:
        return "No results found for this query."
    lines: list[str] = []
    for i, hit in enumerate(hits, 1):
        unit = hit.unit
        excerpt = textwrap.shorten(unit.text, width=200, placeholder="...")
        node_title = titles_by_id.get(unit.node_id or "", "")
        lines.append(
            f"[{i}] score={hit.score:.3f} | "
            f"type={unit.unit_type.value} | "
            f"pages={unit.page_span.start_page}-{unit.page_span.end_page} | "
            f"node={node_title!r}\n    {excerpt}"
        )
    return "\n\n".join(lines)


@tool
def get_page_text(page_number: int) -> str:
    """Get the complete text of a specific logical page (0-indexed).

    Use this after search_document identifies relevant pages.
    """
    for unit in corpus.units:
        if (
            unit.unit_type is RetrievalUnitType.PAGE_TEXT
            and unit.page_span.start_page == page_number
        ):
            return unit.text
    return f"Page {page_number} not found in corpus."


TOOLS = [search_document, get_page_text]
TOOL_MAP = {t.name: t for t in TOOLS}
print(f"Registered tools: {list(TOOL_MAP.keys())}")

Registered tools: ['search_document', 'get_page_text']


## Section 4 — LangGraph Agent Construction

Standard ReAct agent: the LLM decides whether to call tools or produce a final answer. Tool dispatch uses a manual `execute_tools` node with error handling rather than the prebuilt `ToolNode`, and the model client is constructed per call so each agent turn advances the inline Groq key rotation safely.


In [17]:
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages

SYSTEM_PROMPT = (
    "You are a precise document QA assistant backed by NullVector. "
    "Always use search_document to find evidence before answering. "
    "Cite page numbers when referencing content. "
    "If the corpus does not contain enough evidence, say so clearly."
)


class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


def groq_chat_model_name(model_name: str) -> str:
    if model_name.startswith("groq/"):
        return model_name.removeprefix("groq/")
    return model_name


def build_llm() -> ChatOpenAI:
    api_key, label = next_groq_key_selection()
    print(f"Groq rotation -> {label}")
    return ChatOpenAI(
        model=groq_chat_model_name(AGENT_MODEL),
        api_key=api_key,
        base_url=GROQ_BASE_URL,
    ).bind_tools(TOOLS)


def call_model(state: AgentState) -> dict[str, list[BaseMessage]]:
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    llm = build_llm()
    return {"messages": [llm.invoke(messages)]}


def execute_tools(state: AgentState) -> dict[str, list[ToolMessage]]:
    last: AIMessage = state["messages"][-1]
    results: list[ToolMessage] = []
    for tc in last.tool_calls:
        fn = TOOL_MAP.get(tc["name"])
        try:
            output = fn.invoke(tc["args"]) if fn else f"Unknown tool: {tc['name']}"
        except Exception as exc:
            output = f"Tool error: {exc}"
        results.append(ToolMessage(content=str(output), tool_call_id=tc["id"], name=tc["name"]))
    return {"messages": results}


def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END


graph = StateGraph(AgentState)
graph.add_node("agent", call_model)
graph.add_node("tools", execute_tools)
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")

memory = MemorySaver()
app = graph.compile(checkpointer=memory)

print(f"LangGraph agent compiled successfully with {len(GROQ_API_KEYS)} Groq keys")


LangGraph agent compiled successfully with 5 Groq keys


In [18]:
one_shot_thread_counter = 0


def next_one_shot_thread_id() -> str:
    global one_shot_thread_counter
    one_shot_thread_counter += 1
    return f"pdf-book-one-shot-{one_shot_thread_counter:03d}"


def ask(question: str, *, thread_id: str | None = None, verbose: bool = False) -> str:
    """Submit a question to the agent.

    Omit thread_id for a fresh one-shot conversation.
    Reuse thread_id only when you intentionally want multi-turn memory.
    """
    resolved_thread_id = thread_id or next_one_shot_thread_id()
    config = {"configurable": {"thread_id": resolved_thread_id}}
    result = app.invoke({"messages": [HumanMessage(content=question)]}, config)
    messages = result["messages"]

    if verbose:
        print("=" * 60)
        print("TRACE")
        print("=" * 60)
        print(f"THREAD_ID: {resolved_thread_id}")
        for msg in messages:
            role = type(msg).__name__
            content = textwrap.shorten(str(msg.content), width=120, placeholder="...")
            print(f"  [{role}] {content}")
        print("=" * 60)

    last = messages[-1]
    return last.content if isinstance(last, AIMessage) else str(last)

## Section 5 — Single-Turn QA Demo

The agent searches the corpus, retrieves evidence, and produces a grounded answer.

In [24]:
answer = ask(
    "What topic does the section titled 'Sets' introduce?",
    verbose=True,
)
print()
print("ANSWER:")
print(answer)

Groq rotation -> slot-04 (gsk_...x09a)


[RetrievalSearchStarted] query="section titled 'Sets' topic" document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 limit=5
[RetrievalSearchCompleted] query="section titled 'Sets' topic" document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 limit=5 candidates=3141 hits=5 widened=false


Groq rotation -> slot-05 (gsk_...sDUk)
TRACE
THREAD_ID: pdf-book-one-shot-004
  [HumanMessage] What topic does the section titled 'Sets' introduce?
  [AIMessage] I need to search for information about the section titled 'Sets'. I will use the search_document function to find...
  [ToolMessage] [1] score=27.000 | type=node_text | pages=11-11 | node='Sets' Sets If we can definitely and clearly decide the...
  [AIMessage] The section titled 'Sets' introduces the concept of sets, which is a collection of objects that can be clearly and...

ANSWER:
The section titled 'Sets' introduces the concept of sets, which is a collection of objects that can be clearly and definitely decided. It covers topics such as types of sets, Venn diagrams, equal sets, subsets, universal sets, intersection and union of sets, and the number of elements in a set. (page 10-11)


## Section 6 — Multi-Turn Conversation

Using the same `thread_id` preserves conversation context across turns. The agent can reference earlier answers and retrieve additional evidence on follow-ups.

In [20]:
THREAD = "pdf-book-sets-demo"

print("Q1:", "What does the Sets section introduce?")
a1 = ask("What does the Sets section introduce?", thread_id=THREAD)
print("A1:", a1)
print()

print("Q2:", "Give one example of a set mentioned there.")
a2 = ask("Give one example of a set mentioned there.", thread_id=THREAD)
print("A2:", a2)
print()

print("Q3:", "Is there a practice set in that chapter?")
a3 = ask("Is there a practice set in that chapter?", thread_id=THREAD)
print("A3:", a3)

Q1: What does the Sets section introduce?
Groq rotation -> slot-03 (gsk_...74oG)


[RetrievalSearchStarted] query="Sets section" document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 limit=5
[RetrievalSearchCompleted] query="Sets section" document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 limit=5 candidates=3141 hits=5 widened=false


Groq rotation -> slot-04 (gsk_...x09a)
A1: The Sets section introduces the concept of a set, which is a collection of unique objects, and various related topics such as equal sets, subsets, universal sets, intersection and union of sets, and the number of elements in a set. (pages 10-15)

Q2: Give one example of a set mentioned there.
Groq rotation -> slot-05 (gsk_...sDUk)


[RetrievalSearchStarted] query="set example" document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 limit=5
[RetrievalSearchCompleted] query="set example" document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 limit=5 candidates=3388 hits=5 widened=false


Groq rotation -> slot-01 (gsk_...AnOM)
A2: An example of a set mentioned is A = {2}, which is a singleton set, or a set consisting of a single element. (page 13)

Q3: Is there a practice set in that chapter?
Groq rotation -> slot-02 (gsk_...xhRt)


[RetrievalSearchStarted] query="practice set chapter" document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 limit=5
[RetrievalSearchCompleted] query="practice set chapter" document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 limit=5 candidates=3141 hits=5 widened=false


Groq rotation -> slot-03 (gsk_...74oG)
A3: Yes, there is a practice set in the chapter, specifically Practice set 1.3, which includes questions related to sets. (page 20)


## Section 7 — Streaming Output

Stream agent responses to see tool calls and intermediate steps as they happen.

In [21]:
config = {"configurable": {"thread_id": "pdf-book-streaming-demo"}}
payload = {"messages": [HumanMessage(content="Summarize the Sets section.")]}

print("=" * 60)
print("STREAMING")
print("=" * 60)
for event in app.stream(payload, config, stream_mode="values"):
    last = event["messages"][-1]
    role = type(last).__name__
    if isinstance(last, AIMessage) and last.tool_calls:
        for tc in last.tool_calls:
            print(f"  [{role}] calling {tc['name']}({tc['args']})")
    elif isinstance(last, ToolMessage):
        excerpt = textwrap.shorten(last.content, width=100, placeholder="...")
        print(f"  [{role}] {last.name} -> {excerpt}")
    elif isinstance(last, AIMessage):
        print(f"\n  [{role}] FINAL ANSWER:")
        print(f"  {last.content}")
print("=" * 60)

STREAMING
Groq rotation -> slot-04 (gsk_...x09a)


[RetrievalSearchStarted] query="Sets section" document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 limit=5


  [AIMessage] calling search_document({'query': 'Sets section'})


[RetrievalSearchCompleted] query="Sets section" document=798d2f27d45d2ccda3694005c2ed60bc0b413b8b299f3a5d4ade7c5867094896 limit=5 candidates=3141 hits=5 widened=false


  [ToolMessage] search_document -> [1] score=7.000 | type=node_text | pages=10-10 | node='• Equal sets, subset \t • Universal set' •...
Groq rotation -> slot-05 (gsk_...sDUk)

  [AIMessage] FINAL ANSWER:
  The Sets section appears to cover the basics of set theory, including:

* Definition of a set: a collection of objects that can be clearly defined
* Equal sets: two sets are equal if every element of one set is in the other set and vice versa
* Subsets and universal sets
* Intersection and union of sets
* Number of elements in a set

The section likely includes examples and practice problems to help illustrate these concepts. 

References: pages 10-15, 138.


## Notes

- This notebook requires live Groq credentials through the inline `GROQ_API_KEYS` list — there is no noop fallback because the LangGraph agent needs a live LLM.
- The corpus is loaded from PostgreSQL using the PDF retrieval and tree run IDs from notebook 03. If you used different PDF run IDs, update `PDF_RETRIEVAL_RUN_ID` and `PDF_TREE_RUN_ID`.
- Notebook 04 resolves concrete `manifest_ref` values from the PostgreSQL run tables before loading artifacts. Wildcard refs such as `pg://retrieval/<run_id>/*/run-index.json` are not valid PostgreSQL artifact refs.
- Tool definitions close over the `corpus` and `retrieval_service` objects loaded in Section 2.
- NullVector runtime observability is enabled before the tools are registered, so each retrieval tool call emits progress lines and appends structured events to `artifacts/observability/nullvector-events.jsonl` unless you override that destination with `NULLVECTOR_OBSERVABILITY_JSONL_PATH`.
- `MemorySaver` provides in-process multi-turn memory scoped by `thread_id`. Omit `thread_id` for one-shot questions so the helper generates a fresh thread automatically; reuse `thread_id` only when you intentionally want multi-turn memory.
- Per-call rotation only prints masked key labels such as `slot-03 (gsk_...1234)` so notebook traces never echo the full Groq secrets.